In [1]:
#%%html
#<div style="text-align:center;">
#<iframe src="https://theskylive.com/sky/constellations/" width="1280" height="960"></iframe>
#</div>

## Observable limits on RA & Dec

In [2]:
from astropy.coordinates import EarthLocation, AltAz, SkyCoord
from astropy.time import Time
import astropy.units as u

obs_lat = 32.6 * u.deg
obs_lon = -116.3 * u.deg
obs_hgt = 1131 * u.m
safe_lim = 10 * u.deg
max_mag = 8

# Define observer location
location = EarthLocation.from_geodetic(
    lat=obs_lat, lon=obs_lon, height=obs_hgt
)

# Define observation time
time = Time("2025-06-09 21:30:00")

# Define the celestial object's coordinates (e.g., RA and Dec)
sky_coord = SkyCoord(ra=10 * u.deg, dec=20 * u.deg)

# Create an AltAz frame
altaz_frame = AltAz(obstime=time, location=location)

# Transform the object's coordinates to AltAz
altaz_coord = sky_coord.transform_to(altaz_frame)

# Get the altitude and azimuth
altitude = altaz_coord.alt
azimuth = altaz_coord.az

print(f"Altitude: {altitude}")
print(f"Azimuth: {azimuth}")


# Define location and time
#location = EarthLocation(lat='32.7', lon='-116.33', height=0*u.m)
#obstime = Time.now()
#obstime = datetime.time(21, 0)

# AltAz frame for the observer
#altaz_frame = AltAz(obstime=obstime, location=location)

# Determine Declination range
min_dec = location.lat - 90*u.deg + safe_lim
max_dec = location.lat + 90*u.deg - safe_lim
print(f"Observable Declination range: {min_dec.to_string(unit=u.deg)} to {max_dec.to_string(unit=u.deg)}")


Altitude: 7.19472731138344 deg
Azimuth: 289.3402001357125 deg
Observable Declination range: -47d24m00s to 112d36m00s


In [3]:
obs_zen = 90 * u.deg - obs_lat
safe_min = obs_lat -90 * u.deg + safe_lim
safe_max = obs_lat +90 * u.deg - safe_lim
print(f'The Zenith @SDAA is: {obs_zen:0.2f} deg')
print(f'Safe Declination limits are: {safe_min:0.2f} deg to {safe_max:0.2f} deg')

The Zenith @SDAA is: 57.40 deg deg
Safe Declination limits are: -47.40 deg deg to 112.60 deg deg


In [4]:
#! pip install starcatalogquery

In [5]:
from starcatalogquery import StarCatalog

The EOP file 'finals2000A.all' in /home/jupyter-boyceastro/src/iers/ is already the latest.
The Leap Second file 'Leap_Second.dat' in /home/jupyter-boyceastro/src/iers/ is already the latest.
The JPL Solar System Planetary Ephemeris file 'de440s.bsp' is already in /home/jupyter-boyceastro/src/ephem/.


In [6]:
#sc_raw = StarCatalog.get('at-hyg32') # Get the raw star catalog AT-HYG v3.2
# GET VERSION 4.1 !!!
# https://codeberg.org/astronexus/hyg

In [7]:
#print(sc_raw)

In [8]:
!mkdir -p data_folder

In [9]:
import numpy as np
import pandas as pd
data_folder="data_folder/"
!ls -l {data_folder}
large_datafile_name = data_folder+"/hyg/hyg_v41.csv"
#large_datafile_name = data_folder+"/hyg/one-line.csv"
rdf = pd.read_csv(large_datafile_name,low_memory=False)

total 4980
-rw-r--r-- 1 jupyter-boyceastro jupyter-boyceastro     278 Jun 11 00:52 Exposure_Time.csv
lrwxrwxrwx 1 jupyter-boyceastro jupyter-boyceastro      32 May 19 20:07 hyg -> /srv/data/shared/src/sc-data/hyg
-rw-r--r-- 1 jupyter-boyceastro jupyter-boyceastro      66 Nov 15  2024 readme.md
-rw-r--r-- 1 jupyter-boyceastro jupyter-boyceastro 2604894 Jun 11 01:59 simplified_hyg_catalog.csv
-rw-r--r-- 1 jupyter-boyceastro jupyter-boyceastro   36123 Jun 11 01:04 stars_with_names_hyg_catalog.csv
-rw-r--r-- 1 jupyter-boyceastro jupyter-boyceastro  454707 Jun 11 01:04 tya_simplified_hyg_catalog.csv
-rw-r--r-- 1 jupyter-boyceastro jupyter-boyceastro  183603 Jun 11 01:04 tyb_simplified_hyg_catalog.csv
-rw-r--r-- 1 jupyter-boyceastro jupyter-boyceastro  485238 Jun 11 01:04 tyf_simplified_hyg_catalog.csv
-rw-r--r-- 1 jupyter-boyceastro jupyter-boyceastro  335979 Jun 11 01:04 tyg_simplified_hyg_catalog.csv
-rw-r--r-- 1 jupyter-boyceastro jupyter-boyceastro  847290 Jun 11 01:04 tyk_simplified_hy

In [10]:
#type(rdf)
print(f'StarCatalog contains {rdf.size:0.2e} stars!')

StarCatalog contains 4.43e+06 stars!


In [11]:
stypes = rdf['spect'].unique()
print(f'The catalog contains {stypes.size} distinct spectral types')
stypes

The catalog contains 4310 distinct spectral types


array(['G2V', 'F5', 'K3V', ..., 'K5-M0III', 'A2V:', 'S7.3e'], dtype=object)

In [12]:
fdf = rdf[rdf['spect'].notna()]

In [13]:
print(f'StarCatalog contains {fdf.size:0.2e} filtered stars!')

StarCatalog contains 4.31e+06 filtered stars!


In [14]:
mask = (fdf.ra>=12) & (fdf.ra<=18) & (fdf.dec<=safe_max) & (fdf.dec>=safe_min)  & (fdf.mag<=max_mag)
sdf = fdf[mask]
print(f'StarCatalog contains {sdf.size:0.2e} simplified stars!')

StarCatalog contains 2.71e+05 simplified stars!


In [15]:
# adding more columns to sdf
from scipy.interpolate import interp1d

#x_values = np.array([-2, 0.5, 2.1, 3.45, 4.5, 6.2, 7, 8])
#y_values = np.array([0.3, 0.5, 1.5, 7, 20, 60, 191, 300])

exposure_csv = data_folder+"Exposure_Time.csv"

edf = pd.read_csv(exposure_csv)

x_values = edf['Magnitude'].to_numpy()
y_values = edf['Exposure'].to_numpy()

f = interp1d(x_values, y_values, kind='cubic')

x_interp = sdf['mag']
y_interp = f(x_interp)

# Add a new column 'col3' as the sum of 'col1' and 'col2'
sdf = sdf.assign(exp=y_interp)

sdf = sdf.assign(stype=sdf['spect'])

# Add a new column 'col3' as the sum of 'col1' and 'col2'
#sdf_new = sdf.assign(tgtfn=sdf['tyc']+sdf['proper']+str(sdf['mag'])+str(sdf['spect']))
fname = sdf.apply(lambda x: 'HD' + str(x['hd'])[:-2] + '_Typ' + str(x['spect'])[:2] + '_Mag' + str(x['mag']), axis=1)
sdf = sdf.assign(tgtfn=fname.apply(lambda x: x))

In [16]:
def compute_adj_coord(original_ra_, original_dec_, offset_arcmin_, camera_rotation_deg_): 
    # --- Step 1A: Compute sky position angle for image "left" ---
    original_coord = SkyCoord(original_ra_, original_dec_)
    original_coord_icrs = original_coord.transform_to('icrs')
    
    # --- Step 2: Compute sky position angle for image "left" ---
    sky_PA = (270 - camera_rotation_deg_) * u.deg
    
    # --- Step 3: Offset distance converted to tangent plane components ---
    offset_dist = offset_arcmin_ * u.arcmin
    dx = offset_dist * np.sin(sky_PA)
    dy = offset_dist * np.cos(sky_PA)

    # --- Step 4: Define the offset frame centered on the original target ---
    offset_frame = SkyOffsetFrame(origin=original_coord)

    # --- Step 5: Create a coordinate in the offset frame and transform back ---
    offset_coord = SkyCoord(lon=dx, lat=dy, frame=offset_frame)
    new_coord_ = offset_coord.transform_to('icrs')
    
    return new_coord_

In [17]:
#print(sdf['ra'], sdf['dec'])
#for item1, item2 in zip(sdf['ra'], sdf['dec']):
#    print(item1, item2)

In [18]:
#main code for using SkyOffsetFrame()

import numpy as np
from astropy.coordinates import SkyCoord, SkyOffsetFrame, FK5
from astropy import units as u


# --- User Inputs ---
offset_arcmin = -5                    # Offset distance (arcmin)
camera_rotation_deg = -21.9              # 1/12 of a full rotation = 30° clockwise

# --- Step 1-5 : Call function to calculate adjusted coordinates ---
adjcoord = sdf.apply(lambda x: compute_adj_coord(x['ra']*u.hour, x['dec']*u.deg, offset_arcmin, camera_rotation_deg), axis=1)

# Add the new adjusted ra & dec to dataframe
sdf = sdf.assign(adjra=adjcoord.apply(lambda obj: obj.ra.hour), adjdec=adjcoord.apply(lambda obj: obj.dec.deg))

In [19]:
#[(x.ra.value, x.dec.value) for x in adjcoord]
#sdf

In [20]:
# Add a new adjusted ra & dec to dataframe
#sdf = sdf.assign(adjra=adjcoord.apply(lambda obj: obj.ra*u.hour), adjdec=adjcoord.apply(lambda obj: obj.dec*u.deg))

In [21]:
output_datafile_name = data_folder+"simplified_hyg_catalog.csv"
sdf.to_csv(output_datafile_name)

In [22]:
tya = sdf[sdf['spect'].str.contains(r'^A')]
print(f'StarCatalog contains {tya.size:0.2e} Type A stars!')
output_datafile_name = data_folder+"tya_simplified_hyg_catalog.csv"
tya.to_csv(output_datafile_name)

StarCatalog contains 5.76e+04 Type A stars!


In [23]:
tym = sdf[sdf['spect'].str.contains(r'^M')]
print(f'StarCatalog contains {tym.size:0.2e} Type M stars!')
output_datafile_name = data_folder+"tym_simplified_hyg_catalog.csv"
tym.to_csv(output_datafile_name)

StarCatalog contains 1.43e+04 Type M stars!


In [24]:
tyo = sdf[sdf['spect'].str.contains(r'^O')]
print(f'StarCatalog contains {tyo.size:0.2e} Type O stars!')
output_datafile_name = data_folder+"tyo_simplified_hyg_catalog.csv"
tyo.to_csv(output_datafile_name)

StarCatalog contains 1.01e+03 Type O stars!


In [25]:
tyb = sdf[sdf['spect'].str.contains(r'^B')]
print(f'StarCatalog contains {tyb.size:0.2e} Type B stars!')
output_datafile_name = data_folder+"tyb_simplified_hyg_catalog.csv"
tyb.to_csv(output_datafile_name)

StarCatalog contains 2.25e+04 Type B stars!


In [26]:
tyf = sdf[sdf['spect'].str.contains(r'^F')]
print(f'StarCatalog contains {tyf.size:0.2e} Type F stars!')
output_datafile_name = data_folder+"tyf_simplified_hyg_catalog.csv"
tyf.to_csv(output_datafile_name)

StarCatalog contains 6.19e+04 Type F stars!


In [27]:
tyg = sdf[sdf['spect'].str.contains(r'^G')]
print(f'StarCatalog contains {tyg.size:0.2e} Type G stars!')
output_datafile_name = data_folder+"tyg_simplified_hyg_catalog.csv"
tyg.to_csv(output_datafile_name)

StarCatalog contains 4.24e+04 Type G stars!


In [28]:
tyk = sdf[sdf['spect'].str.contains(r'^K')]
print(f'StarCatalog contains {tyk.size:0.2e} Type K stars!')
output_datafile_name = data_folder+"tyk_simplified_hyg_catalog.csv"
tyk.to_csv(output_datafile_name)

StarCatalog contains 1.07e+05 Type K stars!


In [29]:
prp = sdf[sdf['proper'].notna()]
print(f'StarCatalog contains {prp.size:0.2e} Stars with proper names!')
output_datafile_name = data_folder+"stars_with_names_hyg_catalog.csv"
prp.to_csv(output_datafile_name)

StarCatalog contains 4.16e+03 Stars with proper names!


# Temperature and Peak Wavelength
Quantitatively, the relationship between temperature and peak wavelength of thermal radiation - for a hot plate, a star, or anything else in the universe is:

$$I_{peak} * T = 2.897 * 10^{-3} mK$$

The peak wavelength of radiation can be approximately determined by using obtain a spectral intensity curve from a grating spectrometer calibrated for instrument response. See a curve obtained below:

![tania](TaniaBorealis_Spectrum2.jpg)

In [30]:
### Calculate Temperature of Tania Borealis
### 1 Ang = 10^-10 m
### Peak is observed at 4015 Ang
I_p = 6430e-10 #m
T = 2.897e-3 / I_p
print(f'Approximate Temperature of Tania Borealis is: {T:5e} K')

Approximate Temperature of Tania Borealis is: 4.505443e+03 K
